In [1]:
import pandas as pd
import numpy as np

train    = pd.read_csv("../data/raw/train.csv", parse_dates=["date"])
stores   = pd.read_csv("../data/raw/stores.csv")
items    = pd.read_csv("../data/raw/items.csv")
holidays = pd.read_csv("../data/raw/holidays_events.csv", parse_dates=["date"])
oil      = pd.read_csv("../data/raw/oil.csv", parse_dates=["date"])

print("All files loaded successfully")

C:\Users\innso\AppData\Local\Temp\ipykernel_8952\4109386049.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  train    = pd.read_csv("../data/raw/train.csv", parse_dates=["date"])


All files loaded successfully


In [ ]:
#  File shapes:
print("train:    ", train.shape)
print("stores:   ", stores.shape)
print("items:    ", items.shape)
print("holidays: ", holidays.shape)
print("oil:      ", oil.shape)

train:     (125497040, 6)
stores:    (54, 5)
items:     (4100, 4)
holidays:  (350, 6)
oil:       (1218, 2)


In [3]:
# Key stats:
print("Date range:", train["date"].min(), "to", train["date"].max())
print("Total rows:", f"{len(train):,}")
print("Unique stores:", train["store_nbr"].nunique())
print("Unique items:", train["item_nbr"].nunique())
print("Unique store-item pairs:", train.groupby(["store_nbr","item_nbr"]).ngroups)

Date range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00
Total rows: 125,497,040
Unique stores: 54
Unique items: 4036
Unique store-item pairs: 174685


In [4]:

# Null check:
print("NULLS PER FILE:")
for name, df in [("train", train), ("stores", stores),
                  ("items", items), ("holidays", holidays), ("oil", oil)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f"\n{name}:")
        print(nulls)
    else:
        print(f"\n{name}: No nulls")

NULLS PER FILE:

train:
onpromotion    21657651
dtype: int64

stores: No nulls

items: No nulls

holidays: No nulls

oil:
dcoilwtico    43
dtype: int64


In [5]:
# Sales distribution:
print("unit_sales stats:")
print(train["unit_sales"].describe())
print(f"\nNegative sales rows: {(train['unit_sales'] < 0).sum():,}")
print(f"Zero sales rows:     {(train['unit_sales'] == 0).sum():,}")
print(f"Zero % of total:     {(train['unit_sales'] == 0).mean()*100:.1f}%")

unit_sales stats:
count    1.254970e+08
mean     8.554865e+00
std      2.360515e+01
min     -1.537200e+04
25%      2.000000e+00
50%      4.000000e+00
75%      9.000000e+00
max      8.944000e+04
Name: unit_sales, dtype: float64

Negative sales rows: 7,795
Zero sales rows:     0
Zero % of total:     0.0%


These steps are for **exploring and understanding your data** before building any forecasting model. Here's what each step gives us:

| Step | What it does | Why we need it |
|------|-----------|--------------|
| **Load files** | Reads all CSVs into memory | Confirms data is accessible and in correct format |
| **Check shapes** | Shows rows × columns for each file | Tells us data size and which tables have what fields |
| **Key stats** | Date range, unique stores/items, store-item pairs | Understands the problem scope — how many products across how many stores over what time period |
| **Null check** | Finds missing/empty values | Missing data breaks models; we need to know where to fill or drop |
| **Sales distribution** | Min, max, average sales; negative/zero counts | Reveals data quality issues (negative sales = returns, zeros = out-of-stock) |

**What we get from this:**
- A clear picture of what we're working with
- Knowledge of data quality problems before they break our model
- The foundation for all cleaning, feature engineering, and modeling steps

**Without this exploration, we'd be guessing** — and guessing leads to models that fail silently or give nonsense predictions.

Run the cells and share the output. Then I'll tell you exactly what to clean and how to build your first forecast.


### File Shapes
```
train:     125,497,040 rows × 6 columns   ← 125M transactions, this is your core data
stores:    54 rows × 5 columns            ← 54 stores, small lookup table
items:     4,100 rows × 4 columns         ← 4100 products, small lookup table
holidays:  350 rows × 6 columns           ← 350 holiday/event records
oil:       1,218 rows × 2 columns         ← daily oil prices, one row per day
```

### Nulls — 3 Things to Note

**1. `onpromotion` has 21.6M nulls in train**

That's 17% of all rows. This means for those transactions, we don't know if the item was on promotion. We'll fill these with `0` — assume not on promotion when unknown. This is a modeling assumption you need to be able to explain.

**2. `oil` has 43 nulls**

Oil price wasn't recorded on 43 days — likely weekends and holidays when markets are closed. We'll forward fill these — meaning we carry the last known price forward. This is the only valid approach because we can't use future prices to fill past gaps.

**3. train has zero nulls except onpromotion**

Good. Date, store, item, and sales columns are all complete.

### Sales Distribution — 2 Important Things

**1. Negative sales exist — 7,795 rows**

Negative sales are product returns. They are not demand. We clip these to 0 before modeling. If we left them in, our lag features would carry negative values forward and confuse the model.

**2. Zero sales = 0 rows**

This seems surprising — but it's because Favorita's dataset only records days when a transaction happened. Days with no sales simply don't appear as rows. This matters for feature engineering later.



In [6]:
# Cell 6 — Understand train columns:
print("TRAIN columns and sample:")
display(train.head(5))

print("\nonpromotion unique values:")
print(train["onpromotion"].value_counts(dropna=False))

TRAIN columns and sample:


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN



onpromotion unique values:
onpromotion
False    96028767
NaN      21657651
True      7810622
Name: count, dtype: int64


In [7]:
# Cell 7 — Understand holidays file:
print("HOLIDAYS sample:")
display(holidays.head(10))

print("\nHoliday types:")
print(holidays["type"].value_counts())

print("\nLocale breakdown:")
print(holidays["locale"].value_counts())

print("\nTransferred holidays:")
print(holidays["transferred"].value_counts())


HOLIDAYS sample:


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
5,2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,False
6,2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,False
7,2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False
8,2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False
9,2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False



Holiday types:
type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

Locale breakdown:
locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

Transferred holidays:
transferred
False    338
True      12
Name: count, dtype: int64



### onpromotion column
```
False    96,028,767    ← item was NOT on promotion
NaN      21,657,651    ← unknown, we treat as False
True      7,810,622    ← item WAS on promotion
```
**What to do:** Fill NaN with `False`. Assumption: if promotion status unknown, treat as no promotion. State this in your DECISIONS.md later.

### Holidays file — this is more complex than it looks

```
type:
Holiday       221    ← actual public holidays
Event          56    ← earthquakes, elections, special events
Additional     51    ← extra days added around holidays
Transfer       12    ← holiday moved to different day
Bridge          5    ← connecting day between holiday and weekend
Work Day        5    ← a normally-off day declared as work day

locale:
National    174    ← affects ALL 54 stores
Local       152    ← affects only stores in that specific city
Regional     24    ← affects only stores in that state/province
```

**What this means for modeling:**
- A National holiday affects every store — easy to apply
- A Local holiday only affects stores in that city — need to match with stores.csv city column
- `transferred = True` means that holiday was moved to another day — ignore these 12 rows, the actual holiday appears as a Transfer type row on the new date


In [8]:
# Cell 8 — Understand stores file:

In [9]:
print("STORES sample:")
display(stores)

print("\nStore types:")
print(stores["type"].value_counts())

print("\nClusters:")
print(stores["cluster"].value_counts().sort_index())

print("\nCities (sample):")
print(stores["city"].unique())

STORES sample:


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4
5,6,Quito,Pichincha,D,13
6,7,Quito,Pichincha,D,8
7,8,Quito,Pichincha,D,8
8,9,Quito,Pichincha,B,6
9,10,Quito,Pichincha,C,15



Store types:
type
D    18
C    15
A     9
B     8
E     4
Name: count, dtype: int64

Clusters:
cluster
1     3
2     2
3     7
4     3
5     1
6     6
7     2
8     3
9     2
10    6
11    3
12    1
13    4
14    4
15    5
16    1
17    1
Name: count, dtype: int64

Cities (sample):
['Quito' 'Santo Domingo' 'Cayambe' 'Latacunga' 'Riobamba' 'Ibarra'
 'Guaranda' 'Puyo' 'Ambato' 'Guayaquil' 'Salinas' 'Daule' 'Babahoyo'
 'Quevedo' 'Playas' 'Libertad' 'Cuenca' 'Loja' 'Machala' 'Esmeraldas'
 'Manta' 'El Carmen']


What You Now Know About Your Data
train:    125M transactions, 2013–2017, 54 stores, 4100 items
stores:   54 stores across 22 cities, 5 types (A-E), 17 clusters
items:    4100 products across families
holidays: National/Local/Regional — must match to store city
oil:      Daily oil price, 43 gaps on weekends/market holidays
Key decisions already made:

onpromotion NaN → fill with False
oil nulls → forward fill only
transferred = True holidays → ignore
Local holidays → match on store city
Regional holidays → match on store state
Negative sales → clip to 0